https://qiita.com/nonakayasuo/items/1144471414ab01521b34




In [ ]:
import os
from dotenv import load_dotenv
load_dotenv('../config.env')
os.environ["OPENAI_API_KEY"] = os.getenv('OPENAI_KEY')

import sqlite3
import os
from datetime import datetime

from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer

import chromadb
chroma_client = chromadb.PersistentClient(path="./chroma_db")


/Users/user/Documents/hangul/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [30]:
text_data = ""

# Netflixフォルダ内のテキストファイルを取得
folder_path = '../netflix'
text_files = [f for f in os.listdir(folder_path) if f.endswith('.tsv')]

for file_name in text_files:
    file_path = os.path.join(folder_path, file_name)
    with open(file_path, 'r', encoding='utf-8') as file:
        text_data = text_data + file.read()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=200
)

docs = text_splitter.create_documents([text_data])

# 埋め込みモデル
embedding_model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print("ドキュメントからテキストを抽出")
texts = [doc.page_content for doc in docs]

print("ベクトルに変換")
embeddings = embedding_model.encode(texts)

# 各ドキュメントにIDを付ける
ids = [f"id_{i}" for i in range(len(texts))]

if "rag_sample" in [c.name for c in chroma_client.list_collections()]:
    chroma_client.delete_collection(name="rag_sample")
collection = chroma_client.get_or_create_collection(name="rag_sample")

print("ベクトルを保存")
collection.add(
    documents=texts,
    embeddings=embeddings.tolist(),
    ids=ids
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 12095.74it/s]


ドキュメントからテキストを抽出
ベクトルに変換
ベクトルを保存


In [34]:
# ユーザーの質問
query = """
저희 측에서 원료 납품 재계약 조건을 파격적으로 조정해 드렸는데 고민을 좀 해 보셨을까요
"""

# クエリをベクトルに変換
query_embedding = embedding_model.encode([query])

# 類似検索（トップ1件を取得）
results = collection.query(
    query_embeddings=query_embedding.tolist(),
    n_results=5
)

for result in results['documents']:
    for line in result:
        print(line)
        print()


59:48:00	그러니까 빨리 가서 막아야 돼	だから早く行って止める
59:50:00	1시간 뒤면 에센스가 전국에 팔려 나가	１時間後には 全国に売られていく
59:54:00	레뚜알이 전량 회수 조치 한다고 해도 100% 다 못 돌아올 거고	回収しようとしても 100％は回収できない
59:58:00	- 해석아, 일단 좀 진정하고 - 형	落ち着け 兄貴
1:00:02	솜이 같은 애가 또 나올지도 모른다고	ソミみたいな被害者が 出るかもしれない
1:00:07	빨리 가 레뚜알 쪽엔 내가 연락할게	急げ レトワールには 俺が連絡する
1:00:49	여보세요	もしもし
1:00:52	미생물이 검출됐다니요?	微生物が検出された？
1:01:22	- C 스튜디오가 어디입니까? - 아, 8층입니다	Ｃスタジオは？ ８階です
1:01:25	저, 잠시만 잠시만요!	待ってください
1:01:47	저기요 담예진 씨 대기실이 어디입니까?	ダム･イェジンさんの 控え室は？

49:26:00	누가 그래요?	ウソもいいところよ
49:28:00	거긴 제품에 돈 쓸 생각이 없어요	開発に お金を使う気がなかった
49:31:00	우리 신제품 그대로 베껴서 원료는 싸구려로 바꾸고	うちの新作を盗み 原料を粗悪品にし
49:34:00	아니, 단가는 또 높이 쳐서 팔아먹더라니까	単価を上げ ボロもうけしてる
49:37:00	잠깐만요 방금 뭐라 그러셨어요?	待ってください 今何て？
49:41:00	원료를 바꿨다고요?	原料を粗悪品に？
49:43:00	네	はい
49:44:00	원료를 뭐 이렇게 비싼 걸 쓰냐면서	“原料が高い”と言われ
49:46:00	마진 챙겨야 하니까 저가로 가자고 하더라고요	マージンが必要だから 安物に替えろと
49:50:00	사업가는 마진부터 챙겨야 되는 거야	実業家は マージンを確保すべきだ
49:53:00	우수가 원료를 바꿔치기했대	ウスが原料をすり替えた
50:06:00	그쪽도 손창호가 짠 판에 발 담근 것 같은데	あなたも ソンにだまされてる

10:25	사라는 기다 매출 올려 달라꼬	売上のために 買ってほしいのさ
10:29	아, 맞나?	そう

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.schema import HumanMessage
from langchain_ollama import ChatOllama

# GPTモデルの初期化
# chat = ChatOpenAI(model="gpt-4o-mini")
chat = ChatOllama(model="qwen2.5-coder:7b")

# ユーザーの質問
query = "サムダルが目指していることは？"

# 外部情報（RAGで取得した文書から）
context = results['documents'][0][0]

prompt_with_context = f"""
次の情報を参考にして質問に答えてください。

情報: {context}

質問: {query}

回答:
"""
response_with_context = chat.invoke([HumanMessage(content=prompt_with_context)])
print(response_with_context.content)


提供された情報によれば、サムダルリ（삼달리）という言葉は酔っ払って話している人の文脈で使われています。ただし、具体的な「サムダルが目指していること」についての直接的な情報はありません。

サムダルリ（삼달리）という言葉は、韓国語で「三八路（サンパウロ）」を指し、「サムダリョン」としても知られています。これは、中国から朝鮮半島にかけての歴史的な交易路です。

この文脈では、話題が「酔っ払っている時にサムダルリを目指していた」という内容であり、「どこへ行きたいのか」や「何を達成したいのか」といった具体的な意図は明示されていません。しかし、一般的にサムダルリが象徴する歴史的・文化的背景から考えると、「貿易や交流を通じて豊かさを求めたり、知識や文化を得ようとした」という解釈が可能です。

結論としては、情報だけではサムダルリの具体的な目標について詳しく答えることはできません。しかし、歴史的背景を考慮すると、貿易や文化交流が関連してくる可能性があります。
